# SPI and SPEI from a prepared Zarr store with xarray and Dask

This notebook walks the end-to-end workflow behind the `climate_indices`
package's tutorial example:

```text
prepared Zarr  ->  inspect and validate (xarray)  ->  compute SPI/SPEI (Dask)
     ->  write Zarr  ->  reopen  ->  maps and time series
```

**Scientific question.** How dry or wet was each month of the 1980-2016 record
relative to the 1981-2010 Calibration Period, for every cell of the example
nClimGrid grid?

**Expected outputs.** A `climate_indices_output.zarr` store holding a 3-month
SPI and a 3-month SPEI per grid cell and month, then spatial maps and a time
series plot read back from that store.

**Learning objectives.** After working through this notebook you will be able
to:

1. Explain the distinct roles of Zarr (storage), xarray (labeled model), and
   Dask (lazy scheduling) in this workflow.
2. Open the prepared store lazily and inspect its variables, coordinates,
   units, masks, and chunks.
3. State and check the input contract that SPI/SPEI correctness depends on.
4. Run the public `climate_indices.spi` and `climate_indices.spei` xarray APIs
   over a Dask-backed dataset and persist the result.

The results here are illustrative: they demonstrate the package's xarray/Dask
path on a reduced sample dataset. They are not a validated drought assessment
of any location.

## Terms used throughout

- **SPI (Standardized Precipitation Index)**: precipitation deficit or surplus
  standardized against the distribution of precipitation at each location and
  calendar month.
- **SPEI (Standardized Precipitation-Evapotranspiration Index)**: the same
  standardization applied to the water balance (precipitation minus potential
  evapotranspiration), so it accounts for atmospheric demand.
- **PET (potential evapotranspiration)**: the water that would evaporate and
  transpire under the given atmospheric conditions, as monthly totals here.
- **Timescale**: the accumulation window, `scale` in code. SPI/SPEI at a
  3-month Timescale compare overlapping 3-month totals; at 12 months they
  compare annual water availability.
- **Periodicity**: the observation frequency, monthly here. Periodicity is
  distinct from Timescale: monthly observations can feed any Timescale.
- **Calibration Period**: the years used to fit the reference distribution
  (1981-2010 in this example). It must cover a complete record for every
  location, otherwise fitted distributions are not comparable across cells.

The indices are dimensionless standardized anomalies, not water amounts.
Negative values mean drier than the Calibration Period, positive means wetter,
and values near zero are near normal.

## Environment and data setup

From the repository root:

```bash
uv sync --group dev
uv run jupyter lab notebooks/zarr_dask_spi_spei.ipynb
```

The `dev` dependency group provides Jupyter, Dask's `distributed`, matplotlib,
and Zarr. The kernel's working directory is this notebook's directory
(`notebooks/`), which is why the default data path below is `../data/e2e`.
This teaching notebook lives in `notebooks/`; the one-time input preparation
support lives in `scripts/`.

### One-time input preparation (optional)

The prepared store is the normal tutorial entry point. To regenerate it:

```bash
uv run --with h5py --with zarr scripts/prepare_e2e_inputs.py
```

- Downloads about 5 MB of source NetCDF, verified against pinned SHA-256
  digests; reruns use the cache under `data/e2e/source/` and need no network
  access.
- Validates the result, then atomically switches `data/e2e/current` to the new
  generation.
- `data/e2e/` is gitignored; never commit it.

### Source attribution

The example inputs are a reduced and modified sample of the NOAA/NCEI
nClimGrid v1 dataset, pinned at `monocongo/example_climate_indices` commit
`ae57c488af832c1ebfdf864c8ed7d16636e2e36f`. Precipitation and PET are monthly
totals over a subset grid and period; the units declared in the source are
relabeled to `mm` only after they are verified, never numerically converted.

- Vose, Russell S., Applequist, Scott, Squires, Mike, Durre, Imke, Menne,
  Matthew J., Williams, Claude N. Jr., Fenimore, Chris, Gleason, Karin, and
  Arndt, Derek (2014): NOAA Monthly U.S. Climate Gridded Dataset (NClimGrid),
  Version 1 (reduced subset). NOAA National Centers for Environmental
  Information. DOI:10.7289/V5SX6B56
- No NOAA endorsement is implied, and this sample is not the unaltered
  dataset.
- Full provenance and redistribution constraints:
  `docs/research/nclimgrid-acquisition-and-redistribution.md`.

## Configuration and data location

One configuration literal drives everything below. The
`CLIMATE_INDICES_E2E_DATA` environment variable overrides the data directory
(the test suite uses this); otherwise the default `../data/e2e` resolves
relative to this notebook.

In [ ]:
import json
import os
import shutil
from pathlib import Path

import pandas as pd
import xarray as xr

from climate_indices import compute, indices, spei, spi
from climate_indices.exceptions import CoordinateValidationError, InvalidArgumentError

# Canonical configuration: matches data/e2e/manifest.json (1980-2016 monthly
# inputs, 1981-2010 Calibration Period).
pipeline_config = {
    "scale": 3,  # 3-Month SPI/SPEI
    "distribution_spi": indices.Distribution.gamma,
    "distribution_spei": indices.Distribution.pearson,
    "data_start_year": 1980,
    "cal_start_year": 1981,
    "cal_end_year": 2010,
    "periodicity": compute.Periodicity.monthly,
}

# scripts/prepare_e2e_inputs.py atomically publishes the generation that
# data/e2e/current points at.
data_root = Path(os.environ.get("CLIMATE_INDICES_E2E_DATA", "../data/e2e"))
if not (data_root / "current").exists():
    raise FileNotFoundError(
        f"Prepared inputs not found: {data_root / 'current'}. "
        "Generate them once with: uv run --with h5py --with zarr scripts/prepare_e2e_inputs.py"
    )
data_dir = (data_root / "current").resolve()
prepared_zarr = data_dir / "cache_prepared_input.zarr"
final_output_zarr = data_root / "climate_indices_output.zarr"

In [ ]:
manifest_path = data_dir / "manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError(
        f"Input manifest not found: {manifest_path}. "
        "Regenerate the inputs with: uv run --with h5py --with zarr scripts/prepare_e2e_inputs.py"
    )
manifest = json.loads(manifest_path.read_text())
manifest

## Three complementary layers: Zarr, xarray, Dask

- **Zarr** is chunked array storage on disk. `cache_prepared_input.zarr` holds
  the numbers; on its own it does not know what "lat" or a month means beyond
  array shape.
- **xarray** is the labeled model on top. A `Dataset` is a container of named
  variables that share coordinates; a `DataArray` is one variable with its
  coordinates and attributes; a plain NumPy array carries values and shape
  only. Labels are what let `xr.align`, `.sel`, and the index functions match
  precipitation to PET by coordinate instead of by position.
- **Dask** is lazy scheduling. An operation on a Dask-backed `DataArray` only
  builds a task graph; the tasks run when you call `.compute()`, `.load()`, or
  write with `.to_zarr()`.

The roles are complementary, not interchangeable: Zarr without xarray loses
labels and metadata, xarray without Dask would materialize the whole grid, and
Dask without either would schedule unlabeled blocks.

## Open the prepared store lazily

`xr.open_zarr(..., consolidated=True)` reads the store's consolidated metadata
and returns a `Dataset` whose variables are Dask arrays: nothing is loaded into
memory yet. The store's own layout is one chunk along `time` — required so
distribution fitting sees each full time series — with 10x10 spatial blocks
written during preparation. Spatial chunking and worker settings are covered
in the Dask execution section below; here we only inspect what already
exists.

In [ ]:
ds = xr.open_zarr(prepared_zarr, consolidated=True)
ds

In [ ]:
ds["precip"]

In [ ]:
print("dims:", ds["precip"].dims)
print("time:", str(ds.time.values[0])[:10], "to", str(ds.time.values[-1])[:10], f"({ds.sizes['time']} monthly steps)")
print("lat:", float(ds.lat.min()), "to", float(ds.lat.max()), f"({ds.sizes['lat']} cells)")
print("lon:", float(ds.lon.min()), "to", float(ds.lon.max()), f"({ds.sizes['lon']} cells)")
print("dtype/units:", ds["precip"].dtype, ds["precip"].attrs.get("units"))
print("existing chunks:", ds["precip"].chunks)

# A reduction materializes only what it needs, not the whole grid.
print(f"Grid-mean monthly precip: {float(ds['precip'].mean('time').compute().mean()):.1f} mm")

## Input contract: required for correctness

Everything the calculation needs from its inputs, and the failure each check
rules out. These are correctness requirements, not performance preferences.

In [ ]:
def _validate_monthly_time(time_values):
    """Return complete month-start or month-end timestamps."""
    try:
        time = pd.DatetimeIndex(time_values)
    except (TypeError, ValueError, OverflowError) as exc:
        raise CoordinateValidationError(
            "Time coordinate must contain supported datetime values.",
            coordinate_name="time",
            reason="not datetime-like",
        ) from exc
    if time.empty:
        raise CoordinateValidationError(
            "Time coordinate must not be empty.", coordinate_name="time", reason="empty coordinate"
        )
    if time.is_month_start.all():
        expected_time = pd.date_range(time[0], periods=time.size, freq="MS")
    elif time.is_month_end.all():
        expected_time = pd.date_range(time[0], periods=time.size, freq=pd.offsets.MonthEnd())
    else:
        expected_time = pd.DatetimeIndex([])
    if not time.equals(expected_time):
        raise CoordinateValidationError(
            "Time coordinate must be a complete, chronological sequence of monthly "
            "month-start or month-end timestamps.",
            coordinate_name="time",
            reason="non-monotonic, irregular, or gapped monthly timestamps",
        )
    return time

**Time axis.** Complete, chronological monthly steps with whole calendar
years, starting in `data_start_year`, and a Calibration Period inside the data
range. A gap or a partial year shifts every later accumulation window and
silently misaligns the calibration fit.

In [ ]:
time = _validate_monthly_time(ds["time"].values)
if time[0].month != 1 or time[-1].month != 12:
    raise CoordinateValidationError(
        "Prepared dataset must cover complete calendar years.",
        coordinate_name="time",
        reason="incomplete first or final year",
    )
data_start_year = pipeline_config["data_start_year"]
if time[0].year != data_start_year:
    raise InvalidArgumentError(
        "pipeline_config['data_start_year'] does not match the prepared dataset's first monthly timestamp.",
        argument_name="data_start_year",
        argument_value=str(data_start_year),
        valid_values=str(time[0].year),
    )
cal_start_year, cal_end_year = pipeline_config["cal_start_year"], pipeline_config["cal_end_year"]
if not (data_start_year <= cal_start_year <= cal_end_year <= time[-1].year):
    raise InvalidArgumentError(
        "Calibration Period must fall within the prepared dataset's covered years.",
        argument_name="cal_start_year/cal_end_year",
        argument_value=f"{cal_start_year}-{cal_end_year}",
        valid_values=f"{data_start_year}-{time[-1].year}",
    )

calibration_observations = int(((time.year >= cal_start_year) & (time.year <= cal_end_year)).sum())
print(f"Calendar years: {time[0].year}-{time[-1].year} ({time.size} monthly steps)")
print(f"Calibration Period: {cal_start_year}-{cal_end_year}, {calibration_observations} monthly observations per cell")

**Coordinates.** Precipitation and PET must be identical before they are
combined. xarray's default behavior for binary operations is to *intersect*
coordinates, so a mismatched axis would silently shrink the result; the
preparation step therefore aligned them with `join="exact"` and SPEI is never
allowed to lean on an inner join.

In [ ]:
precip_exact, pet_exact = xr.align(ds["precip"], ds["pet"], join="exact")
print(f"precip and pet aligned exactly: {precip_exact.sizes} / {pet_exact.sizes}")

**Units.** Both variables must be monthly totals in millimeters, declared as
`units == "mm"`. Relabeling an attribute is not a conversion: a source in
inches or in daily rates would produce wrong index values while looking
perfectly labeled.

In [ ]:
for name in ("precip", "pet"):
    units = ds[name].attrs.get("units")
    if units != "mm":
        raise InvalidArgumentError(
            "Prepared inputs must be monthly totals with units='mm'; relabeling an attribute is not a conversion.",
            argument_name=f"{name}.units",
            argument_value=repr(units),
            valid_values="mm",
        )
    print(f"{name}: units={units!r}, dtype={ds[name].dtype}")

**Masks and zeros.** Missing values are `NaN`, meaningful zeros are data.
Precipitation and PET masks must match exactly, or SPEI combines present PET
with missing precipitation. Cells that are entirely `NaN` carry no data at all;
partially missing cells would bias the fitted distribution.

In [ ]:
precip, pet = ds["precip"], ds["pet"]
precip_mask, pet_mask = precip.isnull().any("time"), pet.isnull().any("time")
if not precip_mask.equals(pet_mask):
    raise InvalidArgumentError(
        "Precipitation and PET masks must match; SPEI would otherwise combine present values with missing ones.",
        argument_name="pet",
        argument_value="mask differs from precip",
        valid_values="identical masks",
    )

all_missing = precip.isnull().all("time")
partially_missing = precip.isnull().any("time") & ~all_missing
print(f"All-missing cells: {int(all_missing.sum())} of {all_missing.size}")
print(f"Partially missing cells: {int(partially_missing.sum())}")
print(f"Zero precipitation observations: {int((precip == 0).sum())} (valid data, not missing)")
print(f"Cells with any missing month: {int(precip_mask.sum())}")

**Timescale gap, time chunking, and manifest agreement.** A Timescale of *n*
months cannot produce a value until *n − 1* accumulation months exist, so the
first *n − 1* values per cell are unavailable. On disk, `time` must remain a
single chunk so each cell's full series reaches the distribution fit
([ADR-0003](../docs/adr/0003-dask-time-dimension-single-chunk.md)). Finally,
the store must agree with the manifest that describes it.

In [ ]:
leading_unavailable = pipeline_config["scale"] - 1
print(f"Timescale {pipeline_config['scale']} months: first {leading_unavailable} values unavailable per cell")

time_chunks = ds["precip"].chunks[0]
print(f"Time chunks: {time_chunks}")
if len(time_chunks) != 1:
    raise CoordinateValidationError(
        "Prepared store must keep time as a single chunk.",
        coordinate_name="time",
        reason="multiple time chunks",
    )

store_chunks = list(ds["precip"].encoding["chunks"])
if dict(ds.sizes) != manifest["dimensions"] or store_chunks != list(manifest["chunks"]):
    raise InvalidArgumentError(
        "Prepared store dimensions or chunks disagree with manifest.json.",
        argument_name="manifest",
        argument_value=f"dims={dict(ds.sizes)}, chunks={store_chunks}",
        valid_values=f"dims={manifest['dimensions']}, chunks={manifest['chunks']}",
    )
print(f"Manifest agrees with the store: {manifest['dimensions']}, chunks {manifest['chunks']}")

### Optional performance tuning

The 10x10 spatial chunking written at preparation time and any Dask worker or
memory settings are tuning choices: they change how fast the calculation runs,
not whether it is correct. The only chunking requirement is the single `time`
chunk checked above. Dask execution mechanics — the client, the task graph,
and when to compute or persist — come next.

In [ ]:
from dask.distributed import Client

# Dashboard disabled so the optional bokeh dependency stays optional.
client = Client(n_workers=4, threads_per_worker=2, memory_limit="4GB", dashboard_address=None)

## Canonical calculation path

SPI and SPEI are computed through the public typed API — `climate_indices.spi` and
`climate_indices.spei` — on Dask-backed `xarray.DataArray`s. Under the hood the adapter
runs `xr.apply_ufunc(..., dask="parallelized")`, so labeled dimensions and coordinates
are preserved and results stay lazy until the deliberate Zarr write below
([ADR-0001](../docs/adr/0001-dual-numpy-xarray-api.md),
[ADR-0002](../docs/adr/0002-multiprocessing-cli-dask-xarray.md)).

Dask parallelizes across spatial chunks: each `lat`/`lon` block of the prepared store is
an independent task, while `time` stays a single chunk so distribution fitting sees the
full series at each location
([ADR-0003](../docs/adr/0003-dask-time-dimension-single-chunk.md)). Worker count alone
is not evidence of parallelism — inspect the chunk layout and task graph instead.
Precipitation and PET were aligned with `join="exact"` at preparation time, so SPEI
never relies on xarray's coordinate intersection.

In [ ]:
ds_calc = ds.transpose("time", "lat", "lon").chunk({"time": -1})

index_kwargs = {
    "scale": pipeline_config["scale"],
    "data_start_year": data_start_year,
    "calibration_year_initial": cal_start_year,
    "calibration_year_final": cal_end_year,
    "periodicity": pipeline_config["periodicity"],
}
spi_da = spi(values=ds_calc["precip"], distribution=pipeline_config["distribution_spi"], **index_kwargs)
spei_da = spei(
    precips_mm=ds_calc["precip"], pet_mm=ds_calc["pet"], distribution=pipeline_config["distribution_spei"], **index_kwargs
)
spi_name, spei_name = f"spi_{pipeline_config['scale']}", f"spei_{pipeline_config['scale']}"
ds_output = xr.Dataset({spi_name: spi_da, spei_name: spei_da}, coords=ds_calc.coords)

In [ ]:
# Write beside the target and swap on success so a failed run leaves a
# previously completed store intact.
tmp_path = final_output_zarr.with_name(final_output_zarr.name + ".tmp")
shutil.rmtree(tmp_path, ignore_errors=True)
# The typed API computes in float64 (xr.apply_ufunc(..., output_dtypes=[float])
# regardless of input dtype); downcast on write only, to keep the on-disk
# footprint at the float32 precision the float32 mm inputs actually carry.
float32_encoding = {"dtype": "float32"}
ds_output.to_zarr(
    tmp_path,
    mode="w",
    zarr_format=2,
    consolidated=True,
    encoding={spi_name: float32_encoding, spei_name: float32_encoding},
)
shutil.rmtree(final_output_zarr, ignore_errors=True)
tmp_path.rename(final_output_zarr)

In [ ]:
out_path = Path(final_output_zarr)

In [ ]:
out_path

In [ ]:
out_ds = xr.load_dataset(out_path)

In [ ]:
out_ds

In [ ]:
spi_index = out_ds[spi_name]
spei_index = out_ds[spei_name]

In [ ]:
spi_index.isel(time=9).plot(
    levels=8,
)

In [ ]:
spei_index.isel(time=9).plot(
    levels=8,
)